SetUp & Data Loading

In [2]:
from huggingface_hub import login
login()

In [3]:
!pip install "datasets<3.0.0" librosa scikit-learn hmmlearn jiwer evaluate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [4]:
from datasets import load_dataset
from transformers import WhisperProcessor
from datasets import Audio
from transformers import WhisperFeatureExtractor
from transformers import WhisperTokenizer
from transformers import WhisperForConditionalGeneration
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer
from transformers import pipeline

import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

import evaluate

In [5]:
sindhi_asr = load_dataset("google/fleurs", "sd_in", trust_remote_code=True)  # for Sindhi
# to download all data for multi-lingual fine-tuning uncomment following line
# fleurs_asr = load_dataset("google/fleurs", "all")

# see structure
print(sindhi_asr)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 3443
    })
    validation: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 426
    })
    test: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 980
    })
})


In [6]:
# Alias so the rest of the code works unchanged
dataset = sindhi_asr
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

In [7]:
!pip install librosa

In [8]:
import librosa
import numpy as np

In [16]:
print(sindhi_asr["train"].column_names)

['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id']


 Feature Extraction

In [9]:
import numpy as np
import librosa


def extract_features(audio_array, sr=16000, n_fft=400, hop_length=160, n_mels=80):

    audio = audio_array.astype(np.float32)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels
    )

    log_mel = librosa.power_to_db(mel, ref=np.max)

    delta = librosa.feature.delta(log_mel)
    delta2 = librosa.feature.delta(log_mel, order=2)

    features = np.vstack([log_mel, delta, delta2]).T

    # reduce memory usage
    features = features.astype(np.float16)

    return features


def prepare_features(split, max_samples=None):

    features_list = []
    labels_list = []

    data = dataset[split]

    if max_samples is not None:
        data = data.select(range(max_samples))

    for i, sample in enumerate(data):

        if i % 200 == 0:
            print(f"Processing {i}/{len(data)}")

        audio = sample["audio"]["array"]
        transcription = sample["transcription"]

        feats = extract_features(audio)

        features_list.append(feats)
        labels_list.append(transcription)

    return features_list, labels_list


# limit dataset size to prevent RAM crash
train_feats, train_labels = prepare_features("train")
test_feats, test_labels   = prepare_features("test")

Processing 0/3443
Processing 200/3443
Processing 400/3443
Processing 600/3443
Processing 800/3443
Processing 1000/3443
Processing 1200/3443
Processing 1400/3443
Processing 1600/3443
Processing 1800/3443
Processing 2000/3443
Processing 2200/3443
Processing 2400/3443
Processing 2600/3443
Processing 2800/3443
Processing 3000/3443
Processing 3200/3443
Processing 3400/3443
Processing 0/980
Processing 200/980
Processing 400/980
Processing 600/980
Processing 800/980


In [10]:
!pip install hmmlearn

 Acoustic Model — GMM-HMM (the classical standard)

In [22]:
import numpy as np
import time
import warnings
from hmmlearn import hmm

warnings.filterwarnings("ignore", category=RuntimeWarning)

MAX_SEGMENTS_PER_CHAR = 500
MIN_SEGMENTS_PER_CHAR = 20


def build_char_hmm_models(
        transcriptions,
        features_list,
        n_components=2,
        n_mix=1,
        n_iter=20):

    print("Step 1/3: Collecting frames per character...")

    char_frames = {}

    for idx, (feats, text) in enumerate(zip(features_list, transcriptions)):

        if idx % 100 == 0:
            print(f"Processed {idx}/{len(features_list)} utterances", end="\r")

        T = len(feats)

        # remove spaces from training targets
        import re
        chars = [c for c in text if re.match(r'[\u0600-\u06FF ]', c)]

        if not chars:
            continue

        num_chars = len(chars)
        avg_len = max(8, T // max(num_chars, 1))

        for ch in chars:
          seg_length = np.random.randint(avg_len//2, avg_len*2)

        if T <= seg_length:
          continue
        start = np.random.randint(0, T - seg_length)
        end = start + seg_length

        segment = feats[start:end]
        char_frames.setdefault(ch, []).append(segment)

    print("\nCharacters found:", len(char_frames))
    print("\nStep 2/3: Training HMM models")

    models = {}
    start_time = time.time()

    for i, (char, segments) in enumerate(char_frames.items()):

        if len(segments) < MIN_SEGMENTS_PER_CHAR:
            print(f"Skipping '{char}' (too few samples: {len(segments)})")
            continue

        if len(segments) > MAX_SEGMENTS_PER_CHAR:
            segments = segments[:MAX_SEGMENTS_PER_CHAR]

        X = np.vstack(segments).astype(np.float32)

        # feature normalization
        mean = X.mean(axis=0)

        # variance floor to avoid degenerate Gaussians
        std = np.maximum(X.std(axis=0), 0.1)

        X = (X - mean) / std

        lengths = [len(s) for s in segments]

        print(f"[{i+1}/{len(char_frames)}] Training '{char}' with {len(segments)} segments")

        model = hmm.GMMHMM(
            n_components=n_components,
            n_mix=n_mix,
            covariance_type="diag",
            n_iter=n_iter,
            verbose=False
        )

        try:
            model.fit(X, lengths)
            models[char] = (model, mean, std)

        except Exception as e:
            print(f"Skipping '{char}' due to training error:", e)

    elapsed = time.time() - start_time

    print("\nFinished training")
    print("Total trained character models:", len(models))
    print(f"Training time: {elapsed:.1f} seconds")

    return models


print("=" * 60)
print("Training GMM-HMM models per character...")
print("=" * 60)

hmm_models = build_char_hmm_models(
    train_labels,
    train_feats,
    n_components=3,
    n_mix=2,
    n_iter=25
)

print("\nTraining complete.")

Training GMM-HMM models per character...
Step 1/3: Collecting frames per character...
Processed 3400/3443 utterances
Characters found: 24

Step 2/3: Training HMM models
[1/24] Training 'ن' with 500 segments
[2/24] Training 'و' with 500 segments
[3/24] Training 'ي' with 500 segments
Skipping 'ڻ' (too few samples: 13)
[5/24] Training 'ا' with 423 segments
Skipping 'ل' (too few samples: 10)
Skipping 'ر' (too few samples: 16)
Skipping 'ڪ' (too few samples: 4)
Skipping ' ' (too few samples: 13)
Skipping 'ت' (too few samples: 8)
Skipping 'ہ' (too few samples: 11)
Skipping 'ب' (too few samples: 2)
Skipping 'ط' (too few samples: 2)
Skipping '۾' (too few samples: 10)
Skipping 'ڙ' (too few samples: 2)
Skipping 'ه' (too few samples: 4)
Skipping 'ِ' (too few samples: 1)
Skipping 'ٽ' (too few samples: 2)
Skipping 'س' (too few samples: 4)
Skipping 'ز' (too few samples: 2)
Skipping '،' (too few samples: 2)
Skipping 'م' (too few samples: 2)
Skipping 'ئ' (too few samples: 2)
Skipping 'ڊ' (too few sampl

In [23]:
def decode_with_hmm_beam(features,
                         models,
                         ngram_counts,
                         context_counts,
                         vocab,
                         chunk_size=10,
                         beam_width=5,
                         lm_weight=0.6):

    chars = list(models.keys())
    T = len(features)

    # beam: list of (sequence, score)
    beam = [("", 0.0)]

    for start in range(0, T, chunk_size):

        chunk = features[start:start + chunk_size]

        if len(chunk) < 5:
            continue

        new_beam = []

        for seq, score in beam:

            for ch in chars:

                model, mean, std = models[ch]

                X = (chunk - mean) / std
                X = np.clip(X, -5, 5).astype(np.float32)

                try:
                    acoustic_score = model.score(X)

                    # repetition penalty
                    if seq and seq[-1] == ch:
                        acoustic_score -= 5

                    candidate = seq + ch

                    lm_score = ngram_score(
                        candidate,
                        ngram_counts,
                        context_counts,
                        vocab
                    )

                    total_score = score + acoustic_score + lm_weight * lm_score

                    new_beam.append((candidate, total_score))

                except:
                    continue

        # keep best hypotheses
        new_beam = sorted(new_beam, key=lambda x: x[1], reverse=True)
        beam = new_beam[:beam_width]

    best_seq = beam[0][0] if beam else ""

    return best_seq

Step 3B: Acoustic Model — SVM / Random Forest on MFCC statistics

In [24]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np


def utterance_features(feats):
    """
    Convert variable-length acoustic sequence into a fixed-size vector.
    """

    feats = feats.astype(np.float32)

    mean = feats.mean(axis=0)

    # variance floor
    std = np.maximum(feats.std(axis=0), 1e-5)

    return np.concatenate([mean, std])


# Build dataset
X_train = np.array([utterance_features(f) for f in train_feats], dtype=np.float32)
X_test  = np.array([utterance_features(f) for f in test_feats], dtype=np.float32)


# Predict first character baseline
y_train = [t[0] if len(t) > 0 else "?" for t in train_labels]
y_test  = [t[0] if len(t) > 0 else "?" for t in test_labels]


svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        kernel="rbf",
        C=10,
        gamma="scale"
    ))
])


print("Training SVM baseline...")
svm_pipeline.fit(X_train, y_train)

acc = svm_pipeline.score(X_test, y_test)

print(f"SVM first-character accuracy: {acc*100:.2f}%")

Training SVM baseline...
SVM first-character accuracy: 15.61%


Step 4: Language Model — n-gram (replaces Whisper's decoder)

In [25]:
from collections import defaultdict
import math


def build_ngram_lm(transcriptions, n=3):

    """
    Train character-level n-gram language model.
    """

    ngram_counts = defaultdict(lambda: defaultdict(int))
    context_counts = defaultdict(int)

    vocab = set()

    for text in transcriptions:

        text = text.lower().strip()

        padded = " " * (n - 1) + text + " "

        for i in range(len(padded) - n + 1):

            context = padded[i:i + n - 1]
            next_ch = padded[i + n - 1]

            ngram_counts[context][next_ch] += 1
            context_counts[context] += 1

            vocab.add(next_ch)

    vocab = sorted(vocab)

    return ngram_counts, context_counts, vocab


def ngram_score(text, ngram_counts, context_counts, vocab, n=3, alpha=0.1):

    """
    Compute log probability of a string under the n-gram LM.
    """

    text = text.lower().strip()

    # pad start and end
    padded = " " * (n - 1) + text + " "

    log_prob = 0.0
    V = len(vocab)

    for i in range(len(padded) - n + 1):

        context = padded[i:i + n - 1]
        ch = padded[i + n - 1]

        count = ngram_counts[context].get(ch, 0)
        total = context_counts.get(context, 0)

        prob = (count + alpha) / (total + alpha * V)

        log_prob += math.log(prob + 1e-12)

    return log_prob


print("Building character 3-gram language model...")

ngram_counts, context_counts, vocab = build_ngram_lm(train_labels, n=3)

print("Vocabulary size:", len(vocab))
print("Unique contexts:", len(ngram_counts))

Building character 3-gram language model...
Vocabulary size: 124
Unique contexts: 1988


Step 5: Evaluation — WER & CER (same metrics as your Whisper code)

In [26]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def normalize_text(text):
    """
    Normalize text for fair comparison.
    """
    text = text.lower().strip()
    text = " ".join(text.split())
    return text


def evaluate_classical(features_list, references, models, max_samples=None):

    predictions = []
    refs_clean = []

    if max_samples is None:
        max_samples = len(features_list)

    for i, (feats, ref) in enumerate(zip(features_list[:max_samples],
                                         references[:max_samples])):

        # Decode prediction
        pred = decode_with_hmm_beam(
        feats,
        models,
        ngram_counts,
        context_counts,
        vocab
)

        # Normalize
        pred = normalize_text(pred)
        ref = normalize_text(ref)

        predictions.append(pred)
        refs_clean.append(ref)

        if i % 20 == 0:
            print(f"{i}/{max_samples}")
            print("REF :", ref[:50])
            print("PRED:", pred[:50])
            print()

    # Compute metrics
    wer = wer_metric.compute(
        predictions=predictions,
        references=refs_clean
    )

    cer = cer_metric.compute(
        predictions=predictions,
        references=refs_clean
    )

    print("\nModel Results")
    print("-------------------------------")
    print(f"WER: {wer*100:.2f}%")
    print(f"CER: {cer*100:.2f}%")

    return wer, cer


# Run evaluation
wer, cer = evaluate_classical(
    test_feats,
    test_labels,
    hmm_models,
    max_samples=50
)

0/50
REF : هوسٽائل ماحول واري ڪورس' لاءِ انٽرنيٽ جي هڪ ڳولا م
PRED: اناااااانانننننونيننانانواناونينواواايويناونييانني

20/50
REF : ڪجهہ فعلن ۽ شين جي وچ ۾ فرق ڪرڻ لاءِ اهو اهم طريقو
PRED: نوووووووايانايوايونويويايواوااووااوايناايوواااويون

40/50
REF : ننڊ ۾ رنڊڪ توهان جي عام ننڊ جي عرصي دوران هرو ڀرو 
PRED: اوييييييننوانينييييييننننينيناننوننناننييانننايوني


Model Results
-------------------------------
WER: 100.00%
CER: 91.64%
